# PoC: Monitoramento Inteligente com VLM & Sumarização
Este notebook é o ambiente de testes para a Prova de Conceito do sistema de inteligência marítima.

## Fluxo da PoC:
1. **Detecção:** YOLOv12 identifica o navio e salva o recorte (crop).
2. **Persistência:** Registro inicial no SQLite com status `pendente`.
3. **VLM (Vision Language Model):** Analisa a imagem salva para identificar tipo, cor e detalhes.
4. **Sumarização:** Consolida as análises em um resumo narrativo do dia.

In [ ]:

import cv2
import yt_dlp
import os
import sqlite3
import pandas as pd
from datetime import datetime
from ultralytics import YOLO
import google.generativeai as genai
from IPython.display import display, Image
from dotenv import load_dotenv

# Carrega as variáveis de ambiente do arquivo .env
load_dotenv()

# setup config
DB_PATH = "poc_maritime.db"
CROPS_DIR = "../assets/detections"
MODEL_PATH = '../assets/yolo12m.pt'

# cria pasta para as imagens
if not os.path.exists(CROPS_DIR):
    os.makedirs(CROPS_DIR)

# inicia o banco de dados SQLite
def init_db():
    with sqlite3.connect(DB_PATH) as conn:
        cursor = conn.cursor()
        
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS detections (
                id INTEGER PRIMARY KEY AUTOINCREMENT, -- 1. Número da linha (gerado sozinho pelo computador)
                track_id INTEGER, -- 2. O 'RG' único que a IA dá para o navio (para não contar o mesmo navio duas vezes)
                timestamp TEXT, -- 3. A data e a hora exata que o navio passou
                image_path TEXT, -- 4. Onde a foto desse navio está guardada no seu disco rígido
                vlm_description TEXT, -- 5. O texto que o Gemini vai escrever sobre o navio depois
                status TEXT DEFAULT 'pendente' -- 6. Se o Gemini já terminou o trabalho dele ou se está na fila
            )
        ''')
        conn.commit()

init_db()

### Configuração da VLM (Google Gemini)
Gemini 1.5 Flash para processar as imagens por ser rápido e gratuito (dentro dos limites da API).

In [ ]:

# Configuro a API do Google usando a chave que salvei no arquivo .env
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

# função de envio para a VLM
def analyze_image_with_vlm(image_path):
    """Envia uma foto para o Google Gemini e recebe uma descrição textual."""
    try:
        model = genai.GenerativeModel('gemini-2.0-flash')
        
        sample_file = genai.upload_file(path=image_path, display_name="Ship Crop")
        
        prompt = "Descreva este navio em uma frase curta, focando no tipo (cargueiro, petroleiro, etc), cor predominante e qualquer identificação visível."
        
        response = model.generate_content([sample_file, prompt])
        
        return response.text.strip()
        
    except Exception as e:
        return f"Erro na análise VLM: {e}"

### 1. Loop de Detecção (YOLOv12)
Célula para capturar alguns navios

tecla de parada 'q'

In [ ]:

# carregando o modelo YOLOv12
yolo_model = YOLO(MODEL_PATH)

url = 'https://www.youtube.com/watch?v=tMYtrEBNVAU'

# pega o link do YouTube e extrair a URL real do vídeo.
ydl_opts = {'format': 'best[height<=480]', 'quiet': True}
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(url, download=False)
    video_url = info['url']

# abre a captura do vídeo aqui.
cap = cv2.VideoCapture(video_url)

seen = set()
count = 0
max_detections = 5

# loop para detectar os navios.
while count < max_detections:
    ret, frame = cap.read()
    if not ret: break

    results = yolo_model.track(frame, conf=0.6, persist=True, verbose=False)

    for result in results:
        if result.boxes.id is not None:
            for box in result.boxes:
                track_id = int(box.id.item())
                cls = int(box.cls.item())
                
                # Se for um barco e eu ainda não vi esse ID, eu salvo o recorte e guardo no banco.
                if result.names[cls] == 'boat' and track_id not in seen:
                    seen.add(track_id)
                    
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    crop_img = frame[y1:y2, x1:x2]
                    
                    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
                    filename = f"poc_boat_{track_id}_{ts}.jpg"
                    crop_path = os.path.abspath(os.path.join(CROPS_DIR, filename))
                    
                    cv2.imwrite(crop_path, crop_img)
                    
                    with sqlite3.connect(DB_PATH) as conn:
                        conn.execute("INSERT INTO detections (track_id, timestamp, image_path) VALUES (?, ?, ?)",
                                     (track_id, ts, filename))
                    
                    print(f"[!] NAVIO IDENTIFICADO! RG: {track_id}. Foto anotada: {filename}")
                    count += 1
    
    # vídeo na tela para eu acompanhar o que está acontecendo.
    cv2.imshow('Monitoramento Porto de Santos - PoC', frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()
print("\nEtapa de detecção concluída com sucesso. Agora vamos para a análise inteligente!")

### 2. Processamento VLM (Vision Language Model)
Vamos percorrer o banco de dados e analisar cada imagem pendente.

In [ ]:

# busca no banco tudo que ainda está pendente de análise.
with sqlite3.connect(DB_PATH) as conn:
    df_pendentes = pd.read_sql_query("SELECT * FROM detections WHERE status = 'pendente'", conn)

    # pra cada imagem, eu chamo o Gemini e atualizo o banco com a descrição que ele me der.
    for idx, row in df_pendentes.iterrows():
        full_image_path = os.path.join(CROPS_DIR, row['image_path'])
        
        print(f"[ANALISANDO] Enviando foto {row['image_path']} para a IA do Google...")
        
        descricao = analyze_image_with_vlm(full_image_path)
        
        with sqlite3.connect(DB_PATH) as conn:
            conn.execute("UPDATE detections SET vlm_description = ?, status = 'concluido' WHERE id = ?",
                         (descricao, row['id']))
        
        print(f"[IA RESPONDEU]: {descricao}\n")

print("O especialista terminou de analisar todas as fotos capturadas!")

### 3. Sumarização do Dia

In [ ]:

# todas as descrições que a IA gerou.
with sqlite3.connect(DB_PATH) as conn:
    df_final = pd.read_sql_query("SELECT vlm_description FROM detections WHERE status = 'concluido'", conn)

todas_descricoes = "\n".join(df_final['vlm_description'].tolist())

# monta um prompt para o Gemini fazer um resumão do dia.
prompt_summary = f"""
Abaixo estão as notas técnicas de todos os navios que passaram pelo Porto de Santos hoje:
{todas_descricoes}

Com base nessas informações, escreva um parágrafo de resumo executivo.
O tom deve ser profissional. Foque em resumir qual foi o principal tipo de movimentação detectada hoje.
"""

# Peço para o modelo gerar o relatório final e imprimo na tela.
model_llm = genai.GenerativeModel('gemini-2.0-flash')
resumo = model_llm.generate_content(prompt_summary)

print("\n" + "="*50)
print("       RELATÓRIO DE INTELIGÊNCIA MARÍTIMA")
print("="*50)
print(resumo.text)
print("="*50)